In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("data/movies_data_cleaned.csv")
df.shape

(1000, 17)

In [2]:
text_cols = ['Genre', 'Subgenre', 'Subgenre 1', 'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'Certificate']

for c in text_cols:
    df[c] = df[c].replace('Unkown', '').fillna('').astype(str).str.strip()

df['content'] = df[text_cols].agg(' '.join, axis=1)

df[['Series_Title', 'content']].head(3)

,Series_Title,content
0,The Shawshank Redemption,Drama Unknown Frank Darabont Tim Robbins Morg...
1,The Godfather,Crime Drama Francis Ford Coppola Marlon Brand...
2,The Dark Knight,Action Crime Drama Christopher Nolan Christian...


In [3]:
tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform(df['content'])

X.shape

(1000, 4329)

In [4]:
sim = cosine_similarity(X, X)
sim.shape

(1000, 1000)

In [5]:
sim[0, 0]

np.float64(1.0000000000000002)

In [6]:
sim[0, :10]  

array([1.        , 0.0049465 , 0.00484771, 0.00508429, 0.00512983,
       0.00491525, 0.00503278, 0.00461104, 0.        , 0.02407777])

In [7]:
def find_title(query, n=5):
    matches = df[df['Series_Title'].str.contains(query, case=False, na=False)]
    return matches[['Series_Title', 'Released_Year', 'IMDB_Rating', 'No_of_Votes']].head(n)

In [8]:
find_title("Fight")

,Series_Title,Released_Year,IMDB_Rating,No_of_Votes
9,Fight Club,1999.0,8.8,1854740
614,The Fighter,2010.0,7.8,340584


In [10]:
def recommend_similar_ml(title, top_n=5, min_votes=100_000):
    idx_list = df.index[df['Series_Title'].str.lower() == title.lower()].tolist()
    
    if not idx_list:
        return f"Movie not found: {title}. Try find_title('{title}')"

    idx = idx_list[0]

    scores = list(enumerate(sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:top_n+1]

    movie_indices = [i[0] for i in scores]

    out = df.loc[movie_indices, ['Series_Title', 'Released_Year', 'IMDB_Rating', 'No_of_Votes']].copy()
    out['similarity'] = [i[1] for i in scores]

    out = out[out['No_of_Votes'] >= min_votes]

    return out

In [12]:
recommend_similar_ml("fight club")

,Series_Title,Released_Year,IMDB_Rating,No_of_Votes,similarity
733,Birdman or (The Unexpected Virtue of Ignorance),2014.0,7.7,580291,0.276772
27,Se7en,1995.0,8.6,1445096,0.275653
628,The Curious Case of Benjamin Button,2008.0,7.8,589160,0.271705
40,American History X,1998.0,8.5,1034705,0.224240
941,25th Hour,2002.0,7.6,169708,0.175254


## Conclusion

This project builds a content-based movie recommender using TF-IDF and cosine similarity.
Movies are represented as text features and compared to identify the most similar items.
The system ranks movies by similarity and returns the most relevant recommendations.